# 🎢 The Dynamics of Interest Rate Risk: Stress-Testing Portfolios Across the Yield Curve 💹
<br>

<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  
  <a href="https://colab.research.google.com/github/PatrickJHess/Volume-Four-Chapter-Four/blob/master/colab/Colab_Defining_Duration_and_Convexity.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Volume-Four-Chapter-Four/master?urlpath=lab/tree/notebooks//Defining_Duration_and_Convexity.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>
  
  <a href="https://patrickjhess.github.io/Volume-Four-Chapter-Four/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>

  Before diving into the analysis, let's set up our environment. The following cells prepares the notebook by importing our required libraries and loading the custom functions we will use to build out our matrices.


:::{important} [ ▼ ] How to use this page: Run, Copy, & Download
:class: dropdown

<ul>
  <li><b>⏻ Run code right here:</b> Click the <b>Power Button</b> icon at the top of the screen to activate <b>Live Code</b>.</li>
  <li><b>📋 Copy code:</b> Hover over any code block and click the <b>Clipboard icon</b> in the top-right corner.</li>
  <li><b>📥 Download this file:</b> Click the <b>Download icon</b> (downward arrow) at the top right of the screen to save this exact notebook to your computer.</li>
</ul>
:::





## Importing libraries, modules, And functions
Modules that are included in the standard Python library are imported. When necessary, other modules or libraries are installed before they are imported. (see Control Statements).
```
# Import OS to interact with local computer operating system
import os
import sys
import requests
from datetime import datetime, date
from types import ModuleType

# Import the NumPy library for numerical operations, commonly aliased as np.
try:
    import numpy as np
except:
    !pip install-q numpy
    import numpy as np

# Import the pandas library for data manipulation and analysis, aliased as pd.
try:
    import pandas as pd
except:
    !pip install -q pandas
    import pandas as pd

# Import the scipy library to find roorts of present value function
try:
  import scipy.optimize as optimize
except:
  !pip install -q scipy
  import scipy.optimize as optimize

# Import Altair library to visualize spot rates and yields to maturity
try:
  import altair as alt
except:
  !pip install -q altair
  import altair as alt
```

In [1]:
# Import OS to interact with local computer operating system
import os
import sys
import requests
from datetime import datetime, date
from types import ModuleType

# Import the NumPy library for numerical operations, commonly aliased as np.
try:
    import numpy as np
except:
    !pip install-q numpy
    import numpy as np

# Import the pandas library for data manipulation and analysis, aliased as pd.
try:
    import pandas as pd
except:
    !pip install -q pandas
    import pandas as pd


# Import Altair library to visualize spot rates and yields to maturity
try:
  import altair as alt
except:
  !pip install -q altair
  import altair as alt

:::{important} ☁️ Cloud-Loading: How In-Memory Modules Work
:class: dropdown

**The Logic:**
Usually, Python looks for modules as `.py` files on your hard drive. Here, we are "tricking" Python into treating a string of text from a URL as a live library.

**The Workflow:**
1. **Fetch:** `requests.get(url)` grabs the raw text of your Python script from Dropbox.
2. **Instantiate:** `ModuleType(module_name)` creates an empty "container" in your computer's RAM.
3. **Execute:** `exec(code, module.__dict__)` runs that text inside the container, turning text into live functions.
4. **Register:** By adding it to `sys.modules`, we tell Python: *"If I try to import this later, don't look on the disk—look right here in the memory."*

**Why do this?**
It makes your notebooks **100% portable**. A user can open this in a brand-new environment, and as long as they have an internet connection, all your custom financial functions will "just work."
:::

In [2]:
## Define the URL of the Python module to be downloaded from Dropbox.
# The 'dl=1' parameter in the URL forces a direct download of the file content.
url= 'https://www.dropbox.com/scl/fi/4y5hjxlfphh1ngvbgo77q/\
module_-basic_concepts_fixed_income.py?rlkey=6oxi7mgka42veaat79hcv8boz&st=87sztshr&dl=1'
module_name='basic_concepts_fixed_income'
# Send an HTTP GET request to the URL and store the server's response.
try:
    response = requests.get(url)
    module = ModuleType(module_name)
    exec(response.content.decode('utf-8'), module.__dict__)
    sys.modules[module_name] = module
    # Now we can import from our in-memory module
    from basic_concepts_fixed_income import(create_payoff_df,
                                            ns_spot_rates,
                                            calc_bond_metrics_2d,
                                            calc_bond_metrics_3d)


    print(f"✅ Successfully imported module from URL.")
    # Assign the FredReader attribute
except requests.exceptions.RequestException as e:
    print(f"❌ Error: Could not fetch module from URL. {e}")
except Exception as e:
    print(f"❌ Error: Failed to execute or import the module. {e}")

✅ Successfully imported module from URL.


## 🧭 Notebook Roadmap
 1. **Create the Payoff DataFrame**: The previous chapter used eight specific bonds identified from FEDInvest; we will use those same bonds here. Rather than downloading them again, the `create_payoff_df` function will efficiently generate our payoff DataFrame.

2. **The Term Structure Scenarios**: Unlike the previous chapter, the calculation of spot rates for each scenario here is vectorized. We will explore six conditions:

  * **Base Case**: A traditional upward-sloping term structure with modest short-term curvature.

  * **Parallel Increase**: A constant slope (parallel shift) where the entire curve is bumped up by 100 basis points. The spread between short and long rates remains unchanged.

  * **Increase Rates, Increasing Slope (Bear Steepener)**: Long-term rates rise dramatically (often due to inflation expectations or bond sell-offs) with a more muted increase in short-term rates, steepening the curve.

  * **Increase Short Rates, Decreasing Slope (Bear Flattener)**: Short-term rates rise dramatically (e.g., due to Fed hikes to SOFR) with a muted effect on long-term rates, flattening the curve.

  * **Negative Slope (Inverted) with a Belly Dip**: A dramatic cut in the long-term rate (beta 0), a positive beta 1 that inverts the curve, and a negative curvature (beta 2) that creates a "dip" in the intermediate years—often viewed as a recession indicator.

  * **Flattening the Curve using Tau**: Instead of changing the starting or ending rates, increasing the tau parameter stretches the curve out horizontally, meaning it takes longer for spot rates to climb to the long-run rate.

3. **Calculating Bond Metrics For A Single Scenario (2D Arrays)**: We calculate the yield to maturity, duration, and convexity for each of the eight bonds under the base case scenario.

  * Define the `calc_bond_metrics_2d` function.

  * Estimate metrics for each bond using the base case term structure.

4. **The Barbell And Bullet Portfolios**: We will demonstrate and compare two distinct investment strategies:

  * The **Barbell** strategy, which targets a specific duration by blending long-term and short-term bonds.

  * The **Bullet** strategy, which focuses on bonds concentrated in a narrow range of maturities.

5. **Calculating Bond Metrics Across All Six Scenarios (3D Arrays)**: We evaluate the Barbell and Bullet portfolios across all six yield curve scenarios.

  * Adding the scenario layer increases the dimensionality of our arrays.

  * We execute a vectorized calculation to generate metrics for all bonds and all scenarios simultaneously.

6. **Visualizing The Results**: Finally, we will graph the data to visualize exactly how these six distinct term structure scenarios impact the risk and return metrics of the Barbell and Bullet portfolios.

### 1.🧮 Constructing the Cash Flow Payoff Matrix
Before we can stress-test our bonds against different yield curve scenarios, we need to map out exactly when and how much each bond pays over its lifetime. This code block builds the foundational payoff matrix for our eight selected Treasury bonds.

Here is what the code is doing step-by-step:

* **Setting the Baseline**: We establish the settlement date (May 15, 2026) as our starting point for valuing the future cash flows.

* **Defining the Universe**: We input the specific maturity dates and coupon rates for our eight chosen bonds, ranging from 1-year to nearly 30-year maturities, and organize them into a pandas DataFrame.

* **Generating the Cash Flows**: We pass these parameters into our custom `create_payoff_df` function, which calculates all future coupon payments and the final principal repayments.

* **Transposing for Vectorization**: We transpose (`.T)` the resulting matrix. This is a crucial data-wrangling step: it sets the unique payment dates as the rows (index) and the individual bonds as the columns. This specific orientation allows us to easily apply vectorized array math to all eight bonds simultaneously in later steps.

In [3]:
# set settlement date
settlement=datetime(2026,5,15)

# define the maturities
maturity=['2027-05-15', '2028-05-15','2029-05-15', '2031-05-15',
'2033-05-15', '2037-02-15','2046-05-15', '2056-02-15']

# define the coupons
coupons=[2.375,2.875,2.375,1.625,
3.375,4.75,2.5,4.75]

# create dataframe from maturities and coupons for create_payoff_df
df=pd.DataFrame({'Coupon':coupons},index=pd.to_datetime(maturity))

# create payoff matrix and transpose it some unique payments are index and maturity dates are columns
df_payoff=create_payoff_df(df,settlement=settlement)
df_payoff.index=df.index
df_payoff=df_payoff.T
display(df_payoff)

,2027-05-15,2028-05-15,2029-05-15,2031-05-15,2033-05-15,2037-02-15,2046-05-15,2056-02-15
2026-08-17,0.0000,0.0000,0.0000,0.0000,0.0000,2.375,0.00,2.375
2026-11-16,1.1875,1.4375,1.1875,0.8125,1.6875,0.000,1.25,0.000
2027-02-16,0.0000,0.0000,0.0000,0.0000,0.0000,2.375,0.00,2.375
2027-05-17,101.1875,1.4375,1.1875,0.8125,1.6875,0.000,1.25,0.000
2027-08-16,0.0000,0.0000,0.0000,0.0000,0.0000,2.375,0.00,2.375
...,...,...,...,...,...,...,...,...
2054-02-17,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.00,2.375
2054-08-17,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.00,2.375
2055-02-16,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.00,2.375
2055-08-16,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.00,2.375


## 2. The Term Structure Scenarios

1. **Vectorize `ns_spot_rates`**: the `ns_spot_rates` function used in previous chapters is vectorized to calculate spot rates for multiple term structures.
2. **`scenarios`**:six term structure scenarios an array of tuples
3. **Calculate Spot Rates `rates`**: a revised version of `ns_spot_rates` that is vectorized to calculate spot rates for multiple term structure scenarios.
4. **Zero Prices `df_zeros`**: Dataframe of unique payoff zero prices for each term structure scenario.
5. **Theoretical Prices**: The `df_zero` DataFrame is multiplied into (@) the `df_payoff` DataFrame.

### 2.1 ⚡ Vectorize the function `ns_spot_rates`

The `ns_spot_rates` function takes estimates, maturity in years, a SOFR rate, and a value for $\tau$ (determines how quickly maturity effects dissipate). The function determines if the estimates are a two-dimensional array or simply the parameters of a single term structure.  If a DataFrame of parameters are passed, it's converted to a NumPy array before checking the dimensionality.

:::{dropdown} 🔍 Click to see revised `ns_spot_rates`
:icon: search


```python
def ns_spot_rates(interim_estimates, mat_years, sofr_rate=None, fixed_tau=0.731):
    """
    Calculates spot rates using a fully vectorized Nelson-Siegel model.
    Accepts either single parameter vectors or (N, columns) simulation matrices.
    Dynamically accommodates tau if included in the estimates.
    """
    # 1. Ensure mat_years is a numpy array and handle division-by-zero
    t = np.array(mat_years, dtype=float)
    t = np.where(t == 0, 1e-8, t)
    
    if isinstance(interim_estimates, pd.DataFrame):
        interim_estimates = interim_estimates.to_numpy()
        
    # 2. Check if we are passing a 2D matrix of simulations
    is_matrix = isinstance(interim_estimates, np.ndarray) and interim_estimates.ndim == 2
    
    if sofr_rate is not None:  # Restricted model
        if is_matrix:
            # Base restricted params: b0, b2 (2 columns)
            b_0 = interim_estimates[:, 0][:, np.newaxis]
            b_2 = interim_estimates[:, 1][:, np.newaxis]
            
            # If 3 columns, the 3rd is tau
            if interim_estimates.shape[1] == 3:
                tau = interim_estimates[:, 2][:, np.newaxis]
            else:
                tau = fixed_tau
        else:
            # Fallback for 1D array
            b_0, b_2 = interim_estimates[0], interim_estimates[1]
            tau = interim_estimates[2] if len(interim_estimates) == 3 else fixed_tau

        # Vectorized math
        spot_rates = (
            b_0
            + (sofr_rate - b_0) * (1 - np.exp(-t/tau)) / (t/tau)
            + b_2 * ((1 - np.exp(-t/tau)) / (t/tau) - np.exp(-t/tau))
        )
        
    else:  # Unrestricted model
        if is_matrix:
            # Base unrestricted params: b0, b1, b2 (3 columns)
            b_0 = interim_estimates[:, 0][:, np.newaxis]
            b_1 = interim_estimates[:, 1][:, np.newaxis]
            b_2 = interim_estimates[:, 2][:, np.newaxis]
            
            # If 4 columns, the 4th is tau (Index 3)
            if interim_estimates.shape[1] == 4:
                tau = interim_estimates[:, 3][:, np.newaxis]
            else:
                tau = fixed_tau            
        else:
            # Fallback for 1D array
            b_0, b_1, b_2 = interim_estimates[0], interim_estimates[1], interim_estimates[2]
            tau = interim_estimates[3] if len(interim_estimates) == 4 else fixed_tau

        # Vectorized math
        spot_rates = (
            b_0
            + b_1 * (1 - np.exp(-t/tau)) / (t/tau)
            + b_2 * ((1 - np.exp(-t/tau)) / (t/tau) - np.exp(-t/tau))
        )

    return spot_rates, t
  ```
:::  

### 2.2 〰️ The Term Structure Scenarios

:::{note} 💡 Building on Previous Concepts
Recall from the previous chapter, we visualized these exact six Nelson-Siegel scenarios to demonstrate the relationship between **Yield to Maturity (YTM)** and the underlying term structure. Rather than just visualizing the curves, we will now apply them to our cash flow matrices to calculate theoretical bond prices under each curve shift.
:::

In [4]:
# 1. Define alternative term structure scenarios using Nelson-Siegel parameters
# each tuple represents (beta_0, beta_1, beta_2, tau)
scenarios =np.array([
    (0.0570, -0.0210, -0.016, 3.5829), # Base Case
    (0.0670, -0.0210, -0.016, 3.5829), # Parallel Increase
    (0.0750, -0.0300, -0.016, 3.5829), # Bear Steepner
    (0.0580, -0.0050, -0.016, 3.5829), # Bear Flattener
    (0.0350,  0.0150, -0.025, 3.5829), # Inverted
    (0.0570, -0.0210, -0.016, 8.0000)  # Flatten Curve
])

# descriptive names corresponding to each parameter tuple above
scenario_names = [
    'Base Case',
    'Parallel Increase',
    'Bear Steepner',
    'Bear Flattener',
    'Inverted',
    'Flatten Curve'
]

### 2.3 📍 Vectorize Calculation Of Spot Rates
Now that we have our term structure scenarios defined and our cash flow dates mapped out, it is time to generate the actual interest rates. We do this by applying the Nelson-Siegel formula to every future payment date across all six scenarios simultaneously.

Here is a breakdown of what the code is doing:

* **Measuring Time (`mat_years`)**: Because interest rates are inextricably tied to time, we first need to know exactly how far away each cash flow is. We calculate the time to maturity (in fractional years) for every unique payment date in our payoff matrix by subtracting our settlement date and dividing by 365.25 to account for leap years.

* **Vectorized Execution (`ns_spot_rates`)**: Calculating these rates one by one using a traditional for loop would be incredibly slow and inefficient. Instead, we pass the entire `mat_years` array and our scenarios directly into the `ns_spot_rates` function.

* **The Output**: By leveraging array math (vectorization), Python instantly generates the spot rates for every single cash flow date across the Base Case, Bear Flattener, Inverted, and all other scenarios in one swift calculation. These spot rates will serve as the exact discount rates we need to price our bonds in the next step.

In [5]:
# 2. Calculate the spot rates for each scenario and unique payment dates
# calculate the time to maturity in years for each payment date (using May 15, 2026 as settlement)
settlement=datetime(2026,5,15)
# calculate the time to maturity in years for each payment date (using May 15, 2026 as settlement)
mat_years = (pd.to_datetime(df_payoff.index) - pd.to_datetime(settlement)).days / 365.25

# vectorized calculation of spot rates
rates,maturities=ns_spot_rates(scenarios,mat_years)

### 2.4 ⚙️ Going from a DataFrame of Spot Rates to a DataFrame of Zero Prices
With our spot rates generated across all scenarios, the next step is to convert those rates into zero prices (often called discount factors). A zero price tells us the present value of exactly \$1 received on a specific future date under a specific scenario.

Here is how the code bridges the gap between rates and prices:
* **Continuous Discounting (np.exp(...))**: To find the present value of future cash flows, the code uses continuous compounding. It applies the mathematical formula for continuous discounting: $e^{-rt}$ (where $r$ is the rates matrix and $t$ is the maturities array).
* **Vectorized Math**: By multiplying the rates by the maturities and feeding the negative result into NumPy's exponential function, Python instantly calculates the discount factors for every single payment date across all six scenarios simultaneously.
* **Structuring the DataFrame**: We wrap the resulting NumPy array back into a pandas DataFrame to keep our data neatly aligned. The rows represent our six distinct scenario_names, and the columns match our unique payment dates (`df_payoff.index`). This creates a perfectly shaped grid of zero prices that is ready to be mapped against our cash flow matrix to calculate the actual bond prices.

In [6]:
# Calculate the DataFrame of zero prices
# discount the spot rates of the rates matrix and create a Dataframe
df_zeros = pd.DataFrame(
    np.exp(-rates * maturities),
    columns=df_payoff.index,
    index=scenario_names
)

### 2.5 💰 Calculating Bond Prices for each Term Structure Scenario
The final step in this phase is to calculate the theoretical prices of our bonds. This is where setting up our matrices so carefully in the previous steps pays off. We do this by taking the dot product (@) of our zero-coupon prices (`df_zeros`) and our bond cash flows (`df_payoff`).

Here is why this single line of code is so powerful:

* **The Math Behind the @**: Our `df_zeros` matrix has dimensions of (Scenarios × Payment Dates), and our `df_payoff` matrix is (Payment Dates × Bonds).

* **Simultaneous Discounting**: The dot product systematically multiplies every single future cash flow by its corresponding zero price (discount factor) and sums them up.

* **The Output**: In a fraction of a second, Python calculates the sum of the present values for all future cash flows, giving us the clean theoretical price for every bond, under every single yield curve scenario, neatly packaged into a new DataFrame.

In [7]:
bond_values=pd.DataFrame(df_zeros@df_payoff,
                         columns=df_payoff.columns,
                         index=scenario_names)
display(bond_values)

,2027-05-15,2028-05-15,2029-05-15,2031-05-15,2033-05-15,2037-02-15,2046-05-15,2056-02-15
Base Case,98.677893,98.201711,95.661408,88.762675,94.335701,102.549762,68.645965,96.055848
Parallel Increase,97.697116,96.296957,92.916883,84.598846,88.620179,94.307612,59.436029,82.930249
Bear Steepner,97.682620,96.092211,92.411461,83.349479,86.497727,90.630429,54.670268,76.275854
Bear Flattener,97.214466,95.678481,92.407262,84.716175,89.566715,96.972985,64.279976,90.146750
Inverted,97.857344,97.451067,95.572863,90.927723,99.359857,113.404115,87.403923,126.221218
Flatten Curve,98.725240,98.407501,96.136368,90.018351,96.640701,106.906645,74.490826,103.810947




### 3  ⚖️ The 2D Baseline (Dates $\times$ Bonds): The calc_bond_metrics_2d function, takes a single array of prices.

* **Our cash flows (df_payoff)**: are a 2D matrix shaped (Dates, Bonds).
* **Our time vector $t$**: is reshaped to a column (Dates, 1).
* **Metrics For A Single Scenario**:When we multiply our yield guesses by $t$, NumPy's broadcasting automatically expands the calculation across the grid, allowing scipy.optimize.newton to sum down the rows (collapsing the dates) and return a 1D array of yields shaped (Bonds,).

:::{dropdown} 🔍 Click to see `calc_bond_metrics_2d`

```python
def calc_bond_metrics_2d(guesses, cash_flows, mat_years, prices):
    """
    Calculates YTM, Duration, and Convexity for a single term structure regime.
    Expects Cash Flows as (Dates x Bonds or Portfolios).
    """
    is_scalar_input = np.isscalar(prices)

    p = np.atleast_1d(prices)
    cf = np.asarray(cash_flows)

    # If a single bond's cash flows are passed as a flat list, stand it up into a column (Dates, 1)
    if cf.ndim == 1:
        cf = cf[:, np.newaxis]

    # Stand the dates up into a column: (Dates, 1)
    t = np.asarray(mat_years)[:, np.newaxis]

    if np.isscalar(guesses):
        guesses = np.full(len(p), guesses, dtype=float)
    else:
        guesses = np.atleast_1d(guesses)

    def ytm_objective(y, cf, t, p):
        # y is (8,). t is (100, 1). y * t automatically broadcasts to (100, 8)
        # Sum down the rows (axis=0) to collapse the dates, leaving (8,) PVs
        pv = np.sum(cf * np.exp(-y * t), axis=0)
        return pv - p

    def ytm_derivative(y, cf, t, p):
        return np.sum(-t * cf * np.exp(-y * t), axis=0)

    # --- STEP 1: Solve for YTM ---
    ytm = optimize.newton(
        func=ytm_objective,
        x0=guesses,
        fprime=ytm_derivative,
        args=(cf, t, p)
    )

    # --- STEP 2: Calculate Risk Metrics ---
    # ytm is (8,). t is (100, 1). discounted_cf becomes (100, 8)
    discounted_cf = cf * np.exp(-ytm * t)

    # Sum down the rows (axis=0) and divide by prices
    duration = np.sum(t * discounted_cf, axis=0) / p
    convexity = np.sum((t**2) * discounted_cf, axis=0) / p

    if is_scalar_input:
        return ytm[0], duration[0], convexity[0]

    return ytm, duration, convexity
    ```
:::

### 3.1 🏃‍♂️ Execution Cell: The 2D Baseline

With our 2D metrics function defined, let's put it to work. We will start by calculating the Yield to Maturity, Duration, and Convexity specifically for our **Base Case** scenario.

In the execution cell below, we:
* **Extract the Prices:** Isolate the 'Base Case' theoretical prices from our `bond_values` DataFrame using `.loc`.
* **Format for Speed:** Convert our payoff DataFrame and price Series into NumPy arrays (`.to_numpy()`) to feed into our vectorized function.
* **Initialize the Optimizer:** Provide a starting YTM guess of 5% (`0.05`) for the root-finding algorithm.
* **Organize the Output:** Collect the resulting arrays into a clean `df_metrics` DataFrame for easy reading.

In [8]:
ytm,duration,convexity=calc_bond_metrics_2d(0.05,
                                            df_payoff.to_numpy(),
                                            mat_years,
                                            bond_values.loc['Base Case'].to_numpy())
df_metrics=pd.DataFrame({'YTM':ytm,
                        'Duration':duration,
                        'Convexity':convexity},
                       index=bond_values.columns)
display(df_metrics.round(3))

,YTM,Duration,Convexity
2027-05-15,0.037,0.999,1.001
2028-05-15,0.038,1.959,3.885
2029-05-15,0.039,2.912,8.635
2031-05-15,0.041,4.808,23.697
2033-05-15,0.043,6.271,42.157
2037-02-15,0.045,8.472,83.589
2046-05-15,0.049,14.805,265.746
2056-02-15,0.050,15.672,357.577


### 3.2 📊 Analyzing the Base Case Metrics
This simple table reveals three fundamental laws of fixed-income pricing that will drive everything in our upcoming stress tests:

1. * **The Normal Yield Curve**: YTM steadily increases from 3.7% at the short end (2027) to 5.0% at the long end (2056). This tells us our baseline scenario is an upward-sloping, "normal" term structure, compensating investors with higher yields for taking on longer-term risk.

2. * **Duration vs. Maturity**: Notice how the Duration column grows, but it doesn't scale 1-to-1 with maturity. The 2056 bond matures in roughly 30 years, but its duration is only ~15.67 years. Because these are coupon-paying bonds, the intermediate cash flows pull the weighted average time of repayment heavily forward.

3. * **The Convexity Explosion**: This is the most crucial column for our strategy. While duration grows linearly, Convexity grows exponentially (roughly as the square of time). Moving from the 2033 bond to the 2056 bond roughly doubles the duration, but it multiplies the convexity by more than 8 times (from ~42 to ~357).

The Takeaway: That massive convexity at the long end of the curve is the "secret sauce" of the Barbell strategy. By heavily weighting our portfolio in that 2056 bond, we are buying a tremendous amount of structural curvature, which will protect our portfolio from massive rate shocks.

## 4. 🛠️ Creating the Barbell and Bullet Portfolios




To set up our stress test, we need to construct two portfolios that have the exact same starting duration but vastly different cash flow distributions.



### 4.1 🥧⌛ Portfolio Weights And Durations
🏋️ **The Barbell Strategy**

To keep things simple, the Barbell portfolio will be an equal-weighted investment distributed at the extreme short and long ends of our curve. We will allocate 20% to each of the following five bonds:
* 1-year, 2-year, and 3-year maturities (The Short End) 20-year and 29.75-year maturities (The Long End)

🎯 The Bullet StrategyThe Bullet portfolio is designed to match the exact duration of the Barbell, but it does so by concentrating all of its capital in the belly of the yield curve. We will allocate our weights as follows:

* 19% in the 5-year bond
* 23% in the 7-year bond
* 58% in the 10-year bond

⚖️ Matching the DurationsBy pulling the individual duration metrics from our baseline table, we can prove that both portfolios start with an identical duration of ~7.269.

**Barbell Duration**:

$$0.20 \times (0.999 + 1.959 + 2.912 + 14.805 + 15.672) \approx 7.269$$

**Bullet Duration**:
$$0.19 \times 4.808 + 0.23 \times 6.271 + 0.58 \times 8.472 \approx 7.269$$

Because both portfolios share the same duration, traditional rules of thumb suggest they carry the same interest rate risk.

### 4.2 📗 Constructing the Portfolio Payoff Profiles

Our next step is the caclulation of the payoff of our two portolios.

* **The Barbell Portfolio**: Allocates capital to the extreme ends of the maturity spectrum—specifically, the short-end (indices 0, 1, 2) and the long-end (indices 6, 7)—leaving the belly of the curve unweighted.

* **The Bullet Portfolio**: Concentrates capital entirely in the intermediate sector or the "belly" of the curve (indices 3, 4, 5).

By applying these weight vectors to our asset payoff matrix (`df_payoff`) and summing across the assets (axis=1), we collapse the individual asset dimensions. This yields a single, consolidated cash flow timeline for each portfolio, allowing us to evaluate their comparative structural payoff profiles.

In [9]:
# Define portfolio weights (normalized allocations across the 8 assets)
barbell_weights = np.array([0.2 if i in [0,1,2,6,7] else 0.0 for i in range(8)])
bullet_weights = np.array([0.0, 0.0, 0.0, 0.19, 0.23, 0.58, 0.0, 0.0])

# Calculate portfolio-level cash flow timelines
barbell_payoffs = (df_payoff * barbell_weights).sum(axis=1)
bullet_payoffs = (df_payoff * bullet_weights).sum(axis=1)

# Combine into a unified payoff DataFrame
df_portfolios_payoff = pd.DataFrame({
    'Barbell': barbell_payoffs,
    'Bullet': bullet_payoffs
})

df_portfolios_payoff

,Barbell,Bullet
2026-08-17,0.4750,1.3775
2026-11-16,1.0125,0.5425
2027-02-16,0.4750,1.3775
2027-05-17,21.0125,0.5425
2027-08-16,0.4750,1.3775
...,...,...
2054-02-17,0.4750,0.0000
2054-08-17,0.4750,0.0000
2055-02-16,0.4750,0.0000
2055-08-16,0.4750,0.0000


## 5 🚀 : Dimensional Expansion—From Single Regimes to Scenario Matrices
Up to this point, we have calculated bond metrics under static, stylized term structures. But as we established earlier, treating duration as a static rule of thumb is dangerous. Real-world yield curves are dynamic, and to truly understand our Barbell and Bullet risk, we need to stress-test them across an entire matrix of shifting Nelson-Siegel scenarios. To do this efficiently, we have to graduate from 2D matrices to 3D arrays. Below, we expands the 2D (matrixe) logic to handle multiple scenarios simultaneously without a single for loop.



### 5.1 🧊 The 3D Leap: (Scenarios $\times$ Dates $\times$ Bonds)
To simulate dynamic curve shifts, we need to process multiple price scenarios at once. If we have 6 different Nelson-Siegel regimes, our price array becomes a 2D matrix shaped (6, 8) (Scenarios $\times$ Bonds).

In `calc_bond_metrics_3d`, we use `np.newaxis` to inject a "Scenarios" dimension into our optimizer that returns the yield to maturites:

* **The Shape**: We promote our yields to a 3D array shaped (Scenarios, 1, Bonds). You can visualize this as stacking multiple 2D matrices on top of each other.
* **The Broadcast**: When we multiply this by our (Dates, 1) time vector and (Dates, Bonds) cash flows, NumPy brilliantly aligns the missing dimensions.
* **The Collapse**: The engine calculates a massive (Scenarios, Dates, Bonds) 3D grid of discounted cash flows in memory. By summing across axis=1 (the Dates), we collapse the timeline, leaving us with a (Scenarios, Bonds) 2D matrix of PVs that perfectly matches our input prices.

This vectorization allows us to calculate exact Yield to Maturity, Duration, and Convexity across thousands of simulated regimes in milliseconds.

### 5.2 📐 Execution Cell: The 3D Baseline

Before we execute the code below, let's quickly recap the dimensional "magic" happening inside this function. Moving from a 2D grid to a 3D grid can be hard to visualize, but it is the key to calculating thousands of scenarios instantly.

```{figure} https://raw.githubusercontent.com/PatrickJHess/Volume-Four-Chapter-Four/master/from_2d_to_3d.png
:width: 500px
:align: center
```
When we use `np.newaxis` to promote our yield guesses into a 3D shape `(Scenarios, 1, Bonds)`, NumPy's broadcasting engine automatically aligns it with our `(Dates, 1)` time vector and `(Dates, Bonds)` cash flows.

You can think of the resulting calculation like a stack of spreadsheets:
* **The Rows (Axis 1):** The time to maturity (Dates).
* **The Columns (Axis 2):** The individual assets (Bonds).
* **The Depth (Axis 0):** The different term structure regimes (Scenarios).

By commanding NumPy to sum across the "Dates" axis (`axis=1`), we collapse the time dimension, leaving us with a clean 2D matrix of `(Scenarios, Bonds)` that perfectly matches the dimensions of our input prices.

:::{dropdown} 🔍 Click to see `calc_bond_metrics_3d`

```python
def calc_bond_metrics_3d(guesses, cash_flows, mat_years, prices):
    """
    Calculates YTM, Duration, and Convexity for a matrix of prices (Scenarios x Bonds).
    Assumes continuous compounding.

    Returns:
        ytm (ndarray): Yield to Maturity
        duration (ndarray): Macaulay/Modified Duration
        convexity (ndarray): Convexity
    """
    cf = np.asarray(cash_flows)              # (100, 8)
    t = np.asarray(mat_years)[:, np.newaxis] # (100, 1)
    p = np.asarray(prices)                   # (6, 8)
    is_1d = False
    if p.ndim == 1:
        is_1d = True
        p = p[np.newaxis, :]  # Temporarily promote to (1, Bonds)
    # Broadcast the initial guess
    if np.isscalar(guesses):
        guesses = np.full_like(p, guesses, dtype=float)
    else:
        guesses = np.broadcast_to(guesses, p.shape)

    def ytm_objective(y, cf, t, p):
        # Insert the Dates dimension in the middle: (6, 1, 8)
        y_3d = y[:, np.newaxis, :]

        # Multiply to get (6, 100, 8), then sum across Dates (axis=1) -> (6, 8)
        pv = np.sum(cf * np.exp(-y_3d * t), axis=1)
        return pv - p

    def ytm_derivative(y, cf, t, p):
        y_3d = y[:, np.newaxis,:]
        return np.sum(-t * cf * np.exp(-y_3d * t), axis=1)

    # --- STEP 1: Solve for YTM ---
    ytm = optimize.newton(
        func=ytm_objective,
        x0=guesses,
        fprime=ytm_derivative,
        args=(cf, t, p)
    )

    # --- STEP 2: Calculate Risk Metrics ---
    # Convert the final YTM grid into 3D: (Scenarios, 8, 1)
    y_3d_final = ytm[:, np.newaxis,:]

    # Calculate the final discounted cash flows matrix: (Scenarios, 8, 100)
    discounted_cf = cf * np.exp(-y_3d_final * t)
    # Calculate Duration and Convexity
    # We divide directly by 'p' (the market prices) because the optimizer
    # just guaranteed that sum(discounted_cf) == p
    duration = np.sum(t * discounted_cf, axis=1) / p
    convexity = np.sum((t**2) * discounted_cf, axis=1) / p

    return ytm, duration, convexity
```
:::

### 5.3 💼 Execution Cell: Calculating Portfolio Values for each Scenario

With our portfolio payoff profiles (`df_portfolios_payoff`) and scenario-specific discount factors (`df_zeros`) established, we can calculate the theoretical value of each portfolio across all simulated interest rate environments.

Instead of looping through each scenario and portfolio individually, we leverage NumPy's highly optimized matrix multiplication operator (@).


In [10]:
# Calculate the portfolio values across all scenarios @ (matrix multiplcation)
portfolio_values=pd.DataFrame(df_zeros@df_portfolios_payoff,
                         columns=df_portfolios_payoff.columns,
                         index=scenario_names)
display(portfolio_values)

,Barbell,Bullet
Base Case,91.448565,98.040982
Parallel Increase,85.855447,91.154837
Bear Steepner,83.426483,88.296527
Bear Flattener,87.945387,92.940749
Inverted,100.901283,105.903422
Flatten Curve,94.314176,101.336702


### 5.4 🚀 Execution Cell: The 3D Leap — Portfolio Metrics Across All Scenarios

We now execute the "3D Leap" by passing our consolidated portfolio data into the vectorized engine calc_bond_metrics_3d. Instead of iterating through individual portfolios or yield curve scenarios, this step calculates the Yield-to-Maturity (YTM), Macaulay/Modified Duration, and Convexity for both the Barbell and Bullet portfolios across all 6 scenarios in a single, lightning-fast execution.

📐 The Dimensional Alignment

The 3D engine expects inputs to align structurally so that it can broadcast them into a unified $(S \times T \times P)$ computation space, where $S = 6$ scenarios, $T = 100$ payment dates, and $P = 2$ portfolios:

1. **Initial Guess (0.05)**: A scalar starting point for the numerical solver.

2. **Portfolio Payoffs (df_portfolios_payoff.to_numpy())**: Shaped as $(T \times P)$ or $(100 \times 2)$.

3. **Maturity Timeline (mat_years)**: Vector of payment times shaped as $(T \times 1)$ or $(100 \times 1)$.

Target Portfolio Values (portfolio_values.to_numpy()): Shaped as $(S \times P)$ or $(6 \times 2)$, representing the market-clearing price of each portfolio under each scenario.



In [12]:
# 3D calculation engine YTM, Duration, and Convexity simultaneously for six scenarios
ytms,durations,convexities=calc_bond_metrics_3d(0.05,
                    df_portfolios_payoff.to_numpy(),
                    mat_years,
                    portfolio_values.to_numpy())

## 6. 📈 Visualizing The Results: A Tale of Two Strategies —
With our portfolio metrics computed across all six scenarios in our high-dimensional engine, we can now visualize how these two classic structural styles—the Barbell and the Bullet—react to shifting yield curve dynamics.

Below, we plot the percentage change in Yield to Maturity (YTM), Duration, and Convexity relative to the Base Case scenario. This visualization reveals the core risk-return trade-offs embedded in yield curve positioning.

In [13]:
import altair as alt
import pandas as pd
import altair as alt

# 1. Setup lists for your axes
scenarios = [
    'Base Case', 'Parallel Increase', 'Bear Steepner',
    'Bear Flattener', 'Inverted', 'Flatten Curve'
]

# Convert timestamps to strings immediately so Altair's JSON serializer is happy
bond_cols = portfolio_values.columns.astype(str)

# 2. Create three separate DataFrames directly from your 2D arrays
df_ytm = pd.DataFrame(ytms, index=scenarios, columns=bond_cols)
df_dur = pd.DataFrame(durations, index=scenarios, columns=bond_cols)
df_cvx = pd.DataFrame(convexities, index=scenarios, columns=bond_cols)

# 3. Calculate Percentage Change from Base Case
# .iloc[0] selects the first row ('Base Case') to use as the divisor
df_ytm_pct = df_ytm.div(df_ytm.iloc[0]) - 1
df_dur_pct = df_dur.div(df_dur.iloc[0]) - 1
df_cvx_pct = df_cvx.div(df_cvx.iloc[0]) - 1

# 4. Create a helper function to build a chart for any given DataFrame
def make_chart(df, metric_title):
    # Move the 'Scenario' index into a standard column
    df_reset = df.reset_index().rename(columns={'index': 'Scenario'})

    # Melt into a tidy format
    df_melted = df_reset.melt(id_vars='Scenario', var_name='Bond', value_name='Value')

    # Build the Altair chart
    chart = alt.Chart(df_melted, title=metric_title).mark_line(point=True, strokeWidth=2).encode(
        x=alt.X('Scenario:N',
                sort=scenarios, # Forces chronological order instead of alphabetical
                axis=alt.Axis(labelAngle=-45)),
        y=alt.Y('Value:Q',
                title='% Change from Base Case',
                axis=alt.Axis(format='%')), # Formats the Y-axis labels as percentages
        color=alt.Color('Bond:N', legend=alt.Legend(title="Bonds")),
        tooltip=[
            alt.Tooltip('Bond:N'),
            alt.Tooltip('Scenario:N'),
            alt.Tooltip('Value:Q', title=f'{metric_title} % Change', format='.2%')
        ]
    ).properties(
        width=200,
        height=300
    )

    return chart

# 5. Generate the three individual charts
chart_ytm = make_chart(df_ytm_pct, "Yield to Maturity")
chart_dur = make_chart(df_dur_pct, "Duration")
chart_cvx = make_chart(df_cvx_pct, "Convexity")

# 6. Concatenate them horizontally using the pipe operator
final_chart = (chart_ytm | chart_dur | chart_cvx).resolve_scale(y='independent')

# Display the chart
final_chart

alt.HConcatChart(...)

## 📉 The Illusion of Static Risk: Decoding the Drift
These three charts are the punchline of our term structure stress tests. Both the Barbell and the Bullet started with the exact same duration in our Base Case, but the moment the yield curve begins to twist and shift, their risk profiles violently diverge.

Here is the story the data is telling us:

* **The YTM Tracks Closely (Left Chart)**: While the yields of both portfolios move in the same general direction depending on the regime, the Bullet tends to experience slightly higher relative yield shifts. However, yield alone doesn't tell us how the prices are reacting.

* **The Bullet is a Sitting Duck (Orange Lines)**: Look at the Duration and Convexity charts. The orange line barely moves. Because the Bullet's cash flows are densely concentrated around a single maturity, its risk metrics are incredibly rigid. Whether the curve steepens, flattens, or inverts, the Bullet's duration remains static. It behaves exactly like a textbook rule of thumb.

* **The Barbell Bends and Adapts (Blue Lines)**: The blue line shows massive structural drift. Because the Barbell's cash flows are stretched across the extreme short and long ends of the curve, it is highly sensitive to the shape of the term structure.

  * **Downside Protection**: In a **Parallel Increase** or **Bear Steepener** (where rising rates hurt prices), the Barbell's duration drops by 10% to 15%. As rates go against it, it organically shortens its duration, shedding risk and protecting capital.

  * **Upside Capture**: In an Inverted scenario (where long-end rates fall, driving prices up), the Barbell's duration extends by 15%. It naturally becomes more sensitive just in time to capture maximum price appreciation.

* **The Takeaway**: This is positive convexity in action. The Barbell uses its dispersed cash flows to actively drift in a favorable direction—shortening when rates rise, and lengthening when rates fall. It proves that matching duration at inception is only half the battle; how that duration behaves when the world changes is what truly dictates your risk.

## 🌪️ The Real-World Twist: 10,000 Simulations of a Fed Pivot and QT
We just saw the Barbell look like the undisputed champion of term structure shifts, organically adjusting its duration to protect capital. But get ready for another surprise. The stylized scenarios we just ran are mathematically clean and symmetrical. The real market is anything but. The six scenarios exaggerate volatility, playing into the hands of the barbell strategy.

In our next notebook, we are leaving the sandbox. We will run 10,000 empirical Monte Carlo simulations, utilizing copulas to capture the messy, real-world correlations between our Nelson-Siegel parameters. And we are going to subject our portfolios to one of the most complex and treacherous macroeconomic environments imaginable: a cut in short-term rates combined with Quantitative Tightening (QT).

Why is this so dangerous?

* **The Short-End Drop**: A central bank rate cut pulls the front of the yield curve down aggressively.

* **The Long-End Spike**: Simultaneously, QT means the central bank is offloading long-dated bonds. This floods the market with supply, actively driving long-term yields up.

This combination creates a violent, non-parallel twisting of the yield curve. The Barbell's secret weapon has been its massive exposure to the 30-year bond—but in a QT environment, that exact maturity is the one taking direct fire. Will the Barbell's convexity advantage be enough to offset the isolated damage at the long end of the curve? Or will the Bullet's heavily concentrated middle-ground suddenly look like the safest place to hide?

**🎲🏁 Let the simulations begin.**